In [1]:
pip install pandas numpy scikit-learn matplotlib plotly dash


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 85.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [dash]3/4 [dash]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
import plotly.express as px
from IPython.display import display

#step 1 - load and filter dataset
file_path = "Sales data - Order Delivered.csv"  #replace with your file
df = pd.read_csv(file_path, low_memory=False)
df = df[df['order_status'] == 'COMPLETED'].copy()
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df_relevant = df[['store_id', 'order_date', 'gross_gmv']].dropna()

#step 2 - aggregate customer behavior
clv = df_relevant.groupby('store_id').agg(
    total_revenue=('gross_gmv', 'sum'),
    order_count=('gross_gmv', 'count'),
    avg_order_value=('gross_gmv', 'mean'),
    first_order=('order_date', 'min'),
    last_order=('order_date', 'max')
).reset_index()

#step 3 - calculate recency, tenure, and CLV
max_date = df_relevant['order_date'].max()
clv['recency_days'] = (max_date - clv['last_order']).dt.days
clv['tenure_days'] = (clv['last_order'] - clv['first_order']).dt.days.fillna(0)
clv['estimated_clv'] = clv['avg_order_value'] * clv['order_count']

#step 4 - RFM Scores
clv['R'] = pd.qcut(clv['recency_days'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
clv['F'] = pd.qcut(clv['order_count'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
clv['M'] = pd.qcut(clv['total_revenue'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
clv['RFM_Score'] = clv['R'].astype(str) + clv['F'].astype(str) + clv['M'].astype(str)

#step 5 - KMeans clustering into 10 segments
features = clv[['estimated_clv', 'order_count', 'recency_days']]
features_scaled = StandardScaler().fit_transform(features)

kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
clv['clv_cluster'] = kmeans.fit_predict(features_scaled)

cluster_map = clv.groupby('clv_cluster')['estimated_clv'].mean().sort_values().reset_index()
cluster_map['segment'] = [f'Segment {i+1}' for i in range(10)]
clv = clv.merge(cluster_map[['clv_cluster', 'segment']], on='clv_cluster')

#step 6 - predict future CLV using linear regression
reg = LinearRegression()
X = clv[['order_count']]
y = clv['estimated_clv']
reg.fit(X, y)
clv['predicted_clv'] = reg.predict(X)

#step 7 - save CSV
clv.to_csv("customer_clv_full_analysis.csv", index=False)

#step 8 - plot charts
fig1 = px.histogram(clv, x='segment', title="Customer Segments by CLV")
fig2 = px.scatter(clv, x='order_count', y='estimated_clv', color='segment', title="CLV vs Order Count")
fig3 = px.box(clv, x='segment', y='recency_days', title="Recency Distribution by Segment")
fig4 = px.scatter(clv, x='order_count', y='predicted_clv', title="Predicted Future CLV")
fig5 = px.histogram(clv, x='RFM_Score', title="RFM Score Frequency")


fig1.show()
fig2.show()
fig3.show()
fig4.show()
fig5.show()


In [3]:
#########TOP 5 Customers #######################
import plotly.express as px

#top 5 highest CLV customers
top_5_summary = clv.sort_values(by='estimated_clv', ascending=False).head(5)
top_5_summary = top_5_summary[['store_id', 'segment', 'estimated_clv', 'order_count', 'recency_days']]

#Top 5 Customers by Estimated CLV
fig_top5 = px.bar(
    top_5_summary.sort_values(by='estimated_clv'),
    x='estimated_clv',
    y='store_id',
    color='segment',
    orientation='h',
    title='Top 5 Customers by Estimated CLV',
    labels={'estimated_clv': 'Estimated CLV', 'store_id': 'Customer ID'}
)
fig_top5.show()


In [4]:
######### REpeated Buyers #########################


#flitering High CLV (above median) AND multiple orders (>1)
high_clv_multi_orders = clv[
    (clv['estimated_clv'] > clv['estimated_clv'].median()) &
    (clv['order_count'] > 1)
]
#scatter plot for high-value repeat buyers
fig_repeat = px.scatter(
    high_clv_multi_orders,
    x='order_count',
    y='estimated_clv',
    color='segment',
    size='recency_days',  #bubble size reflects recency (lower is better)
    hover_data=['store_id', 'recency_days'],
    title='High-Value Repeat Buyers: CLV vs Order Count',
    labels={'order_count': 'Number of Orders', 'estimated_clv': 'Estimated CLV'}
)
fig_repeat.update_traces(marker=dict(sizemode='diameter', sizeref=1))
fig_repeat.show()


#top 10 matching customers
high_clv_multi_orders[['store_id', 'segment', 'estimated_clv', 'order_count', 'recency_days']].head(10)



,store_id,segment,estimated_clv,order_count,recency_days
1,1038PFIi8cdLxyfU7lnERV,Segment 3,19235.613,3,10
2,104DvHKDQRlPuPpCrbRUq1,Segment 3,17303.000,2,13
3,105C7fAuFJehhI8Rjxq2s7,Segment 4,28783.000,3,0
5,107AMOMq2NHOR5P0tw3KS5,Segment 3,38279.000,4,8
9,10DptMbyzPdOdVyEKdYrz,Segment 5,136416.000,10,3
11,10IIYaIzgnFB5AfffemGLh,Segment 4,23347.000,4,1
13,10KpymniiOZravGx0onlse,Segment 3,21805.000,2,9
14,10NWsKvuTAc3LokWssX1z,Segment 4,53600.000,2,0
15,10OO10UgtrGMisjuR2Mlxa,Segment 4,29840.000,6,2
16,10QC8F5yrwJrHejuaEpECb,Segment 4,36528.895,5,0


In [5]:
############################ CHURN RISK CUSTOMERS #################

#thresholds
clv_threshold = clv['estimated_clv'].median()
recency_threshold = clv['recency_days'].quantile(0.75)

#filter: High CLV + High Recency = Churn Risk
churn_risk = clv[
    (clv['estimated_clv'] > clv_threshold) &
    (clv['recency_days'] > recency_threshold)
]


import plotly.express as px

fig_churn = px.scatter(
    churn_risk,
    x='recency_days',
    y='estimated_clv',
    color='segment',
    hover_data=['store_id', 'order_count'],
    title='🛑 Churn Risk Customers: High CLV but Inactive',
    labels={'recency_days': 'Days Since Last Purchase', 'estimated_clv': 'Estimated CLV'}
)
fig_churn.show()
#top churn-risk customers
churn_risk[['store_id', 'segment', 'estimated_clv', 'order_count', 'recency_days']].head(10)

,store_id,segment,estimated_clv,order_count,recency_days
22,10U5QBKNqgptcroKeSJGWa,Segment 1,11154.32,1,25
24,10VK6XQJO2piQv89Wi6FIa,Segment 2,29867.00,3,21
53,10zeWVLFBd0oL15SZRXyX8,Segment 1,30185.00,1,23
57,114pT61fAReVF7Y4lxV7xN,Segment 1,57300.00,3,27
97,11bf3yVSxiCExFha16dsBW,Segment 1,12750.00,2,25
101,11fAEus9d5KA5EPdqtwE37,Segment 1,30700.00,1,27
158,12egyDZg91E3lajkhpdMlZ,Segment 1,11245.00,1,29
191,1390CZZlThoc7kyOntFwIR,Segment 1,24200.00,1,24
264,147TXemKxoSzkro4GZ8rJz,Segment 1,47737.50,1,25
271,14DVK6xalZqiOKvccIq4C,Segment 1,12050.00,2,22
